# Visualize CoolRun Route Alternatives

This notebook visualizes the route analysis output.

Run the route scoring step first, either with:
- `POST /analyze-route` from the FastAPI backend, or
- `notebooks/06_visualize_routes.ipynb` if you are staying fully inside notebooks.

This notebook does not change route scoring logic.

In [ ]:
# Import the libraries we need.
# sys lets us add the project folder to Python's import path.
# Path helps us find files in outputs/.
import json
import sys
from pathlib import Path

# IPython.display lets the notebook show the saved HTML map inline.
from IPython.display import IFrame, display

In [ ]:
# Find the project root folder.
# If this notebook is opened from the notebooks/ folder, the project root is one level up.
# If it is opened from the project root, the current folder is already the project root.
current_dir = Path.cwd()

if (current_dir / ".env").exists():
    project_root = current_dir
else:
    project_root = current_dir.parent

# Add the project root to Python's import path so backend helpers can be imported.
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [ ]:
# Import reusable route visualization helpers.
# The map styling and popups live in backend/app/visualization/route_maps.py.
from backend.app.visualization.route_maps import load_json, save_routes_map

In [ ]:
# Define the files produced by the previous steps.
outputs_dir = project_root / "outputs"
metadata_path = project_root / "data" / "vienna_orthofoto_test_metadata.json"

# This file is created by POST /analyze-route or notebook 06.
scored_routes_path = outputs_dir / "scored_routes.geojson"

# This file is created by notebook 03 or POST /analyze-area.
tree_geojson_path = outputs_dir / "detected_trees.geojson"

# UTCI summary and grid are optional here.
# The map can still be created without UTCI, but route popups will show average UTCI as n/a.
utci_summary_path = outputs_dir / "utci_summary.json"
utci_grid_path = outputs_dir / "utci_with_trees.npy"

# This is the final interactive route map.
output_map_path = outputs_dir / "coolrun_routes_map.html"

In [ ]:
# Check that the required route and tree files exist before trying to visualize them.
if not scored_routes_path.exists():
    raise FileNotFoundError(
        f"Missing scored routes: {scored_routes_path}. Run POST /analyze-route or notebook 06 first."
    )

if not tree_geojson_path.exists():
    raise FileNotFoundError(
        f"Missing detected trees: {tree_geojson_path}. Run notebooks 01-05 or POST /analyze-area first."
    )

if not metadata_path.exists():
    raise FileNotFoundError(
        f"Missing Vienna orthofoto metadata: {metadata_path}. Run notebooks/01b_vienna_orthofoto_test.ipynb first."
    )

metadata = json.loads(metadata_path.read_text(encoding="utf-8"))

print(f"Scored routes: {scored_routes_path}")
print(f"Detected trees: {tree_geojson_path}")
print(f"Vienna orthofoto center: lat={metadata['center']['lat']}, lon={metadata['center']['lon']}")
print(f"UTCI summary available: {utci_summary_path.exists()}")
print(f"UTCI grid available: {utci_grid_path.exists()}")

In [ ]:
# Load the 3 alternative routes and detected tree points.
scored_routes = load_json(str(scored_routes_path))
tree_geojson = load_json(str(tree_geojson_path))
utci_summary = load_json(str(utci_summary_path)) if utci_summary_path.exists() else None

route_count = len(scored_routes.get("features", []))
tree_count = len(tree_geojson.get("features", []))

print(f"Loaded routes: {route_count}")
print(f"Loaded detected trees: {tree_count}")
print(f"Selected route IDs: {scored_routes.get('selected')}")

In [ ]:
# Create and save the interactive Folium map.
# The helper shows:
# - shortest route
# - greenest route
# - coolest route
# - detected tree points
# - start and end points
# - route popups with distance, tree density, average UTCI, and CoolRun score
try:
    saved_map_path = save_routes_map(
        scored_routes=scored_routes,
        tree_geojson=tree_geojson,
        output_path=str(output_map_path),
        utci_summary=utci_summary,
    )
    print(f"Saved route map to: {saved_map_path}")
except ModuleNotFoundError as exc:
    if exc.name == "folium":
        print("Folium is not installed in this notebook kernel.")
        print("Run this in a notebook cell, then rerun this cell: %pip install folium")
    else:
        raise

In [ ]:
# Display the saved HTML map inside the notebook.
# If your notebook environment blocks local HTML iframes, open outputs/coolrun_routes_map.html in a browser.
display(IFrame(src=str(output_map_path), width="100%", height=650))

In [ ]:
# Print route details so the scoring output is easy to inspect in the notebook.
for feature in scored_routes.get("features", []):
    props = feature.get("properties", {})
    print("---")
    print(f"Route: {props.get('id')}")
    print(f"Distance: {props.get('distance_m', 0):.0f} m")
    print(f"Tree density score: {props.get('tree_density', 0):.2f}")
    print(f"Average UTCI: {props.get('average_utci')}")
    print(f"CoolRun score: {props.get('coolrun_score')}")